# Exact reproduction — The SEIR model of infectious diseases

**Source:** Jay Gopalakrishnan, *The SEIR model of infectious diseases*, MTH 271
course notes, Portland State University, 22 April 2020.

**Why this one is unambiguous.** The source is a lecture notebook that prints its
full source code. There are no missing hyperparameters, no unreleased data, and
no ambiguity about what was run — so "reproduce exactly" means exact numerical
agreement with the code as published.

`seir_f`, `seir_f2` and the parameter values below are transcribed verbatim from
the source. Everything else is the check harness.

## Claims under test

| # | Claim in the source |
|---|---|
| T1 | β=1, σ=1, γ=0.1 from (0.99, 0.01, 0, 0) gives a bell-shaped infection curve |
| T2 | Reducing β, raising γ, or lowering σ each flattens or delays the curve |
| T3 | Every equilibrium of the basic model is disease-free (`e = i = 0`) |
| T4 | Influx `a=0.005`, `b=0.001` gives an endemic equilibrium near 5% infected |
| T5 | `R0 = β·s₀/γ`; `R0 < 1` → no outbreak, `R0 > 1` → outbreak |
| T6 | Vaccinating a fraction `v` scales `R0` by `(1 − v)` |

In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

CHECKS = []


def record(check_id, claim, expected, observed, passed):
    CHECKS.append((check_id, claim, expected, observed, bool(passed)))
    print(f"{check_id}  {'PASS' if passed else 'FAIL'}  {claim}")
    print(f"      expected: {expected}")
    print(f"      observed: {observed}")


# Transcribed verbatim from the source's seir_f and seir_f2.
def seir_f(t, y, beta, sigma, gamma):
    s, e, i, r = y
    return np.array([-beta * i * s,
                     -sigma * e + beta * i * s,
                     -gamma * i + sigma * e,
                     gamma * i])


def seir_f2(t, y, beta, sigma, gamma, a, b):
    s, e, i, r = y
    return np.array([-beta * i * s + a,
                     -sigma * e + beta * i * s + b,
                     -gamma * i + sigma * e,
                     gamma * i - (a + b)])


print("numpy", np.__version__)

## T1 — the baseline solve

In [ ]:
BETA, SIGMA, GAMMA = 1.0, 1.0, 0.1
Y0 = [0.99, 0.01, 0.0, 0.0]

sol = solve_ivp(seir_f, [0, 60], Y0, rtol=1e-6, args=(BETA, SIGMA, GAMMA))
i_peak = sol.y[2].max()
t_peak = sol.t[sol.y[2].argmax()]
total = sol.y[:, -1].sum()

bell = sol.y[2][0] < i_peak and sol.y[2][-1] < 0.05 * i_peak and 0 < t_peak < 60
record("T1", "beta=1, sigma=1, gamma=0.1 gives a bell-shaped infection curve",
       "interior peak, decays to near zero, mass conserved",
       f"peak i={i_peak:.4f} at t={t_peak:.2f}; final i={sol.y[2][-1]:.2e}; sum={total:.10f}",
       bell and abs(total - 1.0) < 1e-6)

## T2 — parameter study

In [ ]:
def solve_ei(beta=1.0, sigma=1.0, gamma=0.1, s0=0.99, e0=0.01, i0=0.0, r0=0.0, t1=60):
    s = solve_ivp(seir_f, [0, t1], [s0, e0, i0, r0], rtol=1e-7, args=(beta, sigma, gamma))
    return s.t, s.y[1], s.y[2]


rows = []
for name, kw in [("baseline", {}), ("beta=0.5", dict(beta=0.5)),
                 ("gamma=0.5", dict(gamma=0.5)), ("sigma=0.1", dict(sigma=0.1))]:
    t, e, i = solve_ei(**kw)
    rows.append((name, i.max(), t[i.argmax()]))
    print(f"{name:12s} peak i={i.max():.4f} at t={t[i.argmax()]:6.2f}")

base_peak = rows[0][1]
record("T2", "reducing beta, raising gamma, lowering sigma each flatten or delay the curve",
       "beta=0.5 flatter; gamma=0.5 flatter; sigma=0.1 later peak",
       f"{rows[1][1]:.4f}, {rows[2][1]:.4f}, t={rows[3][2]:.2f}",
       rows[1][1] < base_peak and rows[2][1] < base_peak and rows[3][2] > rows[0][2])

## T3 — every equilibrium is disease-free

In [ ]:
residual = seir_f(0.0, np.array([0.4, 0.0, 0.0, 0.6]), BETA, SIGMA, GAMMA)
long_sol = solve_ivp(seir_f, [0, 400], Y0, rtol=1e-9, args=(BETA, SIGMA, GAMMA))
e_end, i_end = long_sol.y[1, -1], long_sol.y[2, -1]

# At t=400 the solver leaves |e|,|i| ~ 1e-7 and i can be slightly negative: that
# is integration noise at rtol=1e-9, not a surviving infection.
settled = max(abs(e_end), abs(i_end))
record("T3", "every equilibrium of the basic SEIR model is disease-free",
       "RHS residual = 0 at (s, 0, 0, r); long solve drives e, i -> 0",
       f"max|residual|={np.abs(residual).max():.2e}; settled={settled:.2e}",
       np.abs(residual).max() < 1e-15 and settled < 1e-5)

## T4 — endemic equilibrium under influx

In [ ]:
endemic = solve_ivp(seir_f2, [0, 150], Y0, rtol=1e-7, args=(BETA, SIGMA, GAMMA, 0.005, 0.001))
tail = endemic.t >= 120
plateau = float(endemic.y[2][tail].mean())
drift = float(endemic.y[2][tail].max() - endemic.y[2][tail].min())

record("T4", "influx turns the disease-free equilibrium into an endemic one near 5%",
       "i settles at approximately 0.05, not decaying to 0",
       f"mean i over t>=120 is {plateau:.4f} (drift {drift:.4f})",
       0.03 <= plateau <= 0.07 and drift < 0.02)

## T5 and T6 — the R0 threshold and the vaccination threshold

In [ ]:
def seir_r0(beta, gamma, s0=1.0, vaccinated=0.0):
    return beta * s0 * (1.0 - vaccinated) / gamma


observed = []
for name, kw in [("no outbreak", dict(beta=0.6, gamma=1.0, s0=0.9, i0=0.1, e0=0.0)),
                 ("outbreak", dict(beta=1.0, gamma=0.5, s0=0.9, i0=0.1, e0=0.0))]:
    t, e, i = solve_ei(t1=60, **kw)
    ei = e + i
    r0 = seir_r0(kw["beta"], kw["gamma"], kw["s0"])
    grew = ei.max() > ei[0] * 1.05
    observed.append((r0, grew))
    print(f"{name:12s} R0={r0:.3f}  max(e+i)={ei.max():.3f}  grew={grew}")

record("T5", "R0 = beta*s0/gamma separates outbreak from no outbreak",
       "R0=0.54 -> no growth; R0=1.80 -> growth",
       f"R0={observed[0][0]:.2f} grew={observed[0][1]}; "
       f"R0={observed[1][0]:.2f} grew={observed[1][1]}",
       (not observed[0][1]) and observed[1][1])

beta_v, gamma_v, s0_v = 1.0, 0.5, 0.9
v_star = 1.0 - gamma_v / (beta_v * s0_v)
below = seir_r0(beta_v, gamma_v, s0_v, vaccinated=v_star + 1e-6)
above = seir_r0(beta_v, gamma_v, s0_v, vaccinated=v_star - 1e-6)
record("T6", "vaccinating fraction v scales R0 by (1 - v); threshold at R0 = 1",
       f"threshold v* = 1 - gamma/(beta*s0) = {v_star:.4f}",
       f"R0(v*+eps)={below:.6f} < 1 < R0(v*-eps)={above:.6f}",
       below < 1.0 < above)

## Verdict

In [ ]:
from pathlib import Path

verdict = pd.DataFrame(CHECKS, columns=["check", "claim", "expected", "observed", "passed"])
print(f"{int(verdict.passed.sum())}/{len(verdict)} checks passed")

out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
verdict.to_csv(out / "seir_basic_checks.csv", index=False)
print("wrote", out / "seir_basic_checks.csv")
verdict[["check", "passed"]]